Mount Google Drive, Import Libraries, and Set Output Directories

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import chi2
from sklearn.decomposition import PCA

# Mount Google Drive to access project folders
from google.colab import drive
drive.mount('/content/drive')

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

# Define Google Drive folder paths
output_dir = '/content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs'
viz_dir = '/content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/eda_visualizations'

# Create directories if they do not exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(viz_dir, exist_ok=True)

print("Google Drive mounted successfully!")
print(f"Outputs directory: {output_dir}")
print(f"Visualizations directory: {viz_dir}")

Mounted at /content/drive
Google Drive mounted successfully!
Outputs directory: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs
Visualizations directory: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/eda_visualizations


Load Dataset from Online UCI Repository

In [2]:
# UCI Machine Learning Repository online link
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"

# Define the 23 column names
columns = [
    'class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
    'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
    'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
    'stalk-surface-below-ring', 'stalk-color-above-ring',
    'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
    'ring-type', 'spore-print-color', 'population', 'habitat'
]

# Load dataset directly from URL
df = pd.read_csv(url, header=None, names=columns)

print(f"Dataset loaded successfully! Initial shape: {df.shape}")
df.head()

Dataset loaded successfully! Initial shape: (8124, 23)


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,stalk-root,stalk-surface-above-ring,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,e,e,s,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,e,c,s,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,e,c,s,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,e,e,s,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,t,e,s,s,w,w,p,w,o,e,n,a,g


Missing Value Handling (Member 1 - IT25103896)

In [3]:
# In this dataset, missing values are denoted by '?' in the 'stalk-root' column
# Replace '?' with NaN
df['stalk-root'] = df['stalk-root'].replace('?', np.nan)

# Check missing count
missing_count = df.isnull().sum()
print(f"Missing values in 'stalk-root': {missing_count['stalk-root']} ({(missing_count['stalk-root']/len(df))*100:.2f}%)")

# Impute missing values using the statistical mode ('b' = bulbous)
mode_val = df['stalk-root'].mode()[0]
df['stalk-root'] = df['stalk-root'].fillna(mode_val)

print(f"Missing values imputed with mode: '{mode_val}'")
print(f"Total remaining missing values: {df.isnull().sum().sum()}")

# Save Step 1 output to Google Drive
step1_path = os.path.join(output_dir, 'step1_missing_handled.csv')
df.to_csv(step1_path, index=False)
print(f"Saved: {step1_path}")

Missing values in 'stalk-root': 2480 (30.53%)
Missing values imputed with mode: 'b'
Total remaining missing values: 0
Saved: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs/step1_missing_handled.csv


Duplicate Detection & Invariant Feature Removal (Member 2 - IT25103870)

In [4]:
# Check and remove duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")
df = df.drop_duplicates()

# Check for invariant features (columns with only 1 unique value / zero variance)
unique_counts = df.nunique()
print("Columns with 1 unique value:", unique_counts[unique_counts == 1].index.tolist())

# Drop 'veil-type' because all records contain the same value 'p' (zero variance)
if 'veil-type' in df.columns:
    df = df.drop(columns=['veil-type'])
    print("Dropped invariant column: 'veil-type'")

print(f"Shape after Step 2: {df.shape}")

# Save Step 2 output to Google Drive
step2_path = os.path.join(output_dir, 'step2_cleaned.csv')
df.to_csv(step2_path, index=False)
print(f"Saved: {step2_path}")

Duplicate rows found: 0
Columns with 1 unique value: ['veil-type']
Dropped invariant column: 'veil-type'
Shape after Step 2: (8124, 22)
Saved: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs/step2_cleaned.csv


Categorical Encoding (Member 3 - IT25103919)

In [5]:
# Map target variable to binary numbers (e: 0 for Edible, p: 1 for Poisonous)
df['class'] = df['class'].map({'e': 0, 'p': 1})
print("Target encoded: e -> 0 (Edible), p -> 1 (Poisonous)")

# Separate features (X) and target (y)
X = df.drop(columns=['class'])
y = df['class']

# Apply One-Hot Encoding (drop_first=True to avoid dummy variable trap)
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)
print(f"Number of features after One-Hot Encoding: {X_encoded.shape[1]}")

# Combine target and encoded features
df_step3 = pd.concat([y, X_encoded], axis=1)
print(f"Shape after Step 3: {df_step3.shape}")

# Save Step 3 output to Google Drive
step3_path = os.path.join(output_dir, 'step3_encoded.csv')
df_step3.to_csv(step3_path, index=False)
print(f"Saved: {step3_path}")

Target encoded: e -> 0 (Edible), p -> 1 (Poisonous)
Number of features after One-Hot Encoding: 94
Shape after Step 3: (8124, 95)
Saved: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs/step3_encoded.csv


Outlier & Anomaly Analysis (Member 4 - IT25103942)

In [6]:
# Detect anomalies using Isolation Forest
iso = IsolationForest(contamination=0.01, random_state=42)
anomalies = iso.fit_predict(X_encoded)

outlier_count = (anomalies == -1).sum()
print(f"Anomalies detected: {outlier_count} out of {len(X_encoded)}")

# Decision: Retain all outliers to avoid losing rare, deadly poisonous mushroom records
print("Decision: Outliers retained to prevent fatal false-negative predictions.")

df_step4 = df_step3.copy()

# Save Step 4 output to Google Drive
step4_path = os.path.join(output_dir, 'step4_outliers_reviewed.csv')
df_step4.to_csv(step4_path, index=False)
print(f"Saved: {step4_path}")

Anomalies detected: 82 out of 8124
Decision: Outliers retained to prevent fatal false-negative predictions.
Saved: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs/step4_outliers_reviewed.csv


Feature Scaling (Member 5 - IT25103909)

In [7]:
# Standardize features using StandardScaler (mean = 0, unit variance = 1)
scaler = StandardScaler()
X_scaled_arr = scaler.fit_transform(X_encoded)
X_scaled = pd.DataFrame(X_scaled_arr, columns=X_encoded.columns)

# Combine target and scaled features
df_step5 = pd.concat([y.reset_index(drop=True), X_scaled], axis=1)
print(f"Features standardized successfully. Shape: {df_step5.shape}")

# Save Step 5 output to Google Drive
step5_path = os.path.join(output_dir, 'step5_scaled.csv')
df_step5.to_csv(step5_path, index=False)
print(f"Saved: {step5_path}")

Features standardized successfully. Shape: (8124, 95)
Saved: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs/step5_scaled.csv


Feature Selection & Dimensionality Reduction (Member 6 - IT25103931)

In [8]:
# 1. Feature ranking using Chi-Square test (evaluated on non-negative binary X_encoded)
chi2_scores, _ = chi2(X_encoded, y)
feat_importance = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Chi2_Score': chi2_scores
}).sort_values(by='Chi2_Score', ascending=False)

print("Top 5 Features by Chi-Square Score:")
print(feat_importance.head(5).to_string(index=False))

# 2. PCA on standardized features (retain 95% of cumulative explained variance)
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

n_pcs = pca.n_components_
print(f"\nPCA complete: Reduced from {X_scaled.shape[1]} features to {n_pcs} Principal Components.")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.sum() * 100:.2f}%")

# Create Principal Component column names
pc_columns = [f'PC_{i+1}' for i in range(n_pcs)]
df_pca = pd.DataFrame(X_pca, columns=pc_columns)

# Create final processed dataset (Target + Principal Components)
final_df = pd.concat([y.reset_index(drop=True), df_pca], axis=1)

# Save final processed dataset to Google Drive
final_path = os.path.join(output_dir, 'final_processed_dataset.csv')
final_df.to_csv(final_path, index=False)

print(f"\nFinal dataset saved successfully: {final_path}")
print(f"Final Dataset Shape: {final_df.shape}")
final_df.head()

Top 5 Features by Chi-Square Score:
                   Feature  Chi2_Score
                    odor_n 2836.187954
                    odor_f 2321.062308
stalk-surface-above-ring_k 1986.408494
stalk-surface-below-ring_k 1914.371389
               gill-size_n 1636.606833

PCA complete: Reduced from 94 features to 55 Principal Components.
Cumulative explained variance: 95.46%

Final dataset saved successfully: /content/drive/MyDrive/Project/AIML/AIML_Project_Mushroom/results/final/outputs/final_processed_dataset.csv
Final Dataset Shape: (8124, 56)


,class,PC_1,PC_2,PC_3,PC_4,PC_5,PC_6,PC_7,PC_8,PC_9,PC_10,PC_11,PC_12,PC_13,PC_14,PC_15,PC_16,PC_17,PC_18,PC_19,PC_20,PC_21,PC_22,PC_23,PC_24,PC_25,PC_26,PC_27,PC_28,PC_29,PC_30,PC_31,PC_32,PC_33,PC_34,PC_35,PC_36,PC_37,PC_38,PC_39,PC_40,PC_41,PC_42,PC_43,PC_44,PC_45,PC_46,PC_47,PC_48,PC_49,PC_50,PC_51,PC_52,PC_53,PC_54,PC_55
0,1,-3.569857,1.642173,-0.008243,0.663031,2.747854,-1.814180,-0.487260,-0.204625,3.882267,-4.723417,0.479641,-2.342531,0.218197,-2.409712,-2.826769,0.702301,-1.581225,0.441416,0.195800,-0.274882,-1.438998,0.731899,-0.142344,-0.096719,-1.478521e-14,-1.415316,0.882934,-0.333067,-1.111377e-14,-1.418197e-14,0.561893,0.157878,-0.953311,-1.080916,0.206710,-0.977290,1.601704,-0.031012,-1.611510,-0.160556,0.783156,-0.989144,1.622681,-0.452352,-0.434711,0.303394,0.517808,-0.894483,-0.330774,-0.102648,0.001906,-0.111500,0.786001,-0.445454,-0.292655
1,0,-2.801072,1.826973,-2.359163,-0.103375,1.473281,-1.363011,2.507955,-2.184067,-1.431248,-0.791577,-2.395103,0.801281,-1.243346,-0.424375,-0.833250,0.272494,1.188140,-1.260960,-0.606625,0.217721,-0.262004,1.155121,-1.411927,-0.478026,-2.907895e-14,-0.342257,0.642528,-0.600866,-4.855142e-15,-1.636717e-14,-0.255027,0.529859,-0.436772,1.766721,1.431211,-2.298440,1.637523,0.144426,-1.870522,-1.202070,0.493733,0.756437,1.297268,-0.148434,0.751004,0.727147,-1.618777,-1.143955,0.624179,-1.031300,0.054412,-0.101346,-0.670182,-0.595980,0.795179
2,0,-4.035111,1.888053,-2.326093,-0.242543,1.101253,-1.837944,4.710235,-4.102669,-2.277331,-0.582488,-2.102016,0.417203,-2.339711,0.531274,0.641693,-0.755548,1.065954,-1.343238,-1.013913,0.123270,1.195277,0.097854,1.157505,0.409807,2.502363e-14,0.085473,-2.258919,0.005213,1.666450e-14,5.031903e-14,1.260000,-0.537609,0.816742,0.369463,1.756478,-0.310994,0.341992,-0.081369,0.597092,1.876517,-0.667490,0.183150,-0.414529,0.461948,-0.536969,-0.767675,1.270059,1.219869,-0.606472,0.815173,-0.223558,0.085334,0.501287,0.072939,-0.197315
3,1,-3.489184,2.176305,-0.389474,0.709931,1.717598,-1.693748,0.045950,-0.404395,3.574323,-4.385258,1.140315,-1.848258,0.643898,-2.234365,-2.102725,0.290256,-1.277485,0.684360,2.049348,0.132945,-0.933229,-2.846959,2.532709,0.715293,5.977602e-14,-0.665794,-0.427221,-0.049758,3.444029e-15,-3.729981e-15,0.453082,-0.794972,0.725392,-0.719866,0.340680,0.874539,-0.090604,-0.295771,-0.195412,0.566064,-0.019339,-0.980976,0.474698,-0.197376,-0.515422,-0.321755,0.643603,0.423119,-0.525764,-0.632658,-0.060287,-0.182754,0.808683,-0.315957,-0.382247
4,0,-2.546049,1.602254,-0.647209,0.442874,2.700940,-0.603053,-2.659778,1.672747,-0.547369,-0.010292,-1.678930,0.862096,-0.084351,0.535122,-1.371805,0.636734,1.506078,-0.905828,-0.646911,-0.306447,0.375832,0.202288,-1.843283,-0.578529,-4.314155e-14,0.312358,1.183446,0.093802,-7.712237e-15,-1.051138e-14,1.393696,0.687135,-1.442476,-0.510401,0.051698,-1.827120,1.462727,0.135026,-1.035005,-0.512759,-0.117105,1.000950,0.588315,-0.038213,0.136300,0.182458,-0.104288,-0.567870,0.081267,-0.013320,-0.108589,0.102023,-0.394912,-0.343081,0.006325
